<a href="https://www.kaggle.com/code/ssevinc/gradient-boosting-from-scratch-vs-scikit-learn?scriptVersionId=303868391" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

Gradient Boosting from Scratch vs. Scikit-Learn
-

This notebook focuses on predicting individual medical costs using regression models — specifically, gradient boosting. The key highlight of this project is the development of a gradient boosting regressor from scratch, followed by a performance comparison with Scikit-learn's GradientBoostingRegressor.

The goal is to not only build an accurate predictive model, but also gain intuitive understanding of how gradient boosting works under the hood, through hands-on implementation.

***🔍 What's in the Notebook?***

**1. Data Preparation**
Load and explore the insurance dataset
Perform basic preprocessing (one-hot encoding for categorical variables)
Visualize relationships between features (e.g., bmi, smoker, age) and the target charges

**2. Train-Test Split**
Divide the data into training and testing sets (70/30 split)

**3. Gradient Boosting From Scratch**
Start with a base prediction (the mean of charges)
Define a custom decision tree splitting rule using domain knowledge (age, bmi, smoker, etc.)


**4. Visualizing Model Performance**
Plot training and test MAE vs. boosting rounds to detect overfitting or underfitting
Visualize predicted vs. actual charges on a scatter plot
Calculate performance metrics:
MAE (Mean Absolute Error)
R² Score

**5. Sklearn GradientBoostingRegressor**
Train on the same data
Predict and evaluate using the same metrics
Visualize the predictions for side-by-side comparison

**6. Results & Comparison**
Compare the hand-built model to Scikit-learn's model
Analyze MAE and R²

**Why This Notebook Matters**

You'll see how predictions improve step-by-step.

You’ll understand the role of residuals in building the next correction.

And you’ll get a visual, intuitive grasp of how boosting “learns”.

If you enjoy this notebook or find it useful, feel free to ⭐️ upvote and fork it!
-



Step 1: Import Libraries
-
**We begin by importing the necessary libraries for data manipulation, visualization, and model building. We'll also bring in Scikit-learn tools later for benchmarking.**

In [ ]:

import numpy as np
import pandas as pd 
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error
import matplotlib.pyplot as plt
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import r2_score
import warnings
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))



Step 2: Load the Dataset
-
**We load the insurance dataset, which contains demographic information and medical charges. This will be the foundation for our predictive modeling. And display the first few rows to get a feel for the data structure, feature types, and potential issues (like categorical variables).**

In [ ]:
data= pd.read_csv("/kaggle/input/insurance/insurance.csv")
data.head(10)

In [ ]:
data.info()

Step 4: Drop 'region' for Simplicity
-

**To keep our model interpretable and lightweight for manual boosting, we drop the `region` column in this version. Feel free to keep it for a more complex model later.**

In [ ]:
data = data.drop(['region'],axis='columns')

Step 5: One-Hot Encode Categorical Variables
-

**Since models can't directly work with text values like 'sex' or 'smoker', we convert them into numerical columns using one-hot encoding. We drop the first category to avoid redundancy.**


In [ ]:

df_encoded = pd.get_dummies(data, columns=['smoker','sex'],drop_first=True)

In [ ]:
print(df_encoded.info())


Step 6: Visual Exploration (Optional)
-
**This step lets us visually inspect relationships between features and the target variable (`charges`). For example, you’ll notice smokers generally have much higher charges.**

In [ ]:
warnings.filterwarnings("ignore")
sns.pairplot(df_encoded)


Step 7: Train-Test Split
-
**We divide the dataset into training and testing sets. This allows us to evaluate our model’s ability to generalize to unseen data.**

In [ ]:
train_data,test_data = train_test_split(
    df_encoded,test_size=0.3, random_state=42
)

Step 8: Set Base Prediction
-
**To start our boosting model, we first predict the same value for everyone — the mean of training `charges`. This becomes the starting point for improving our predictions via residual corrections.**


In [ ]:
base = np.mean(train_data['charges'])
print(base)

Step 9: Calculate Residuals
-

**Now based on this "prediction" we will calculate the residuals for each data point.**

**residual = (observed value-predicted value)**


In [ ]:
residuals = train_data['charges'] - base
print(residuals)

Step 10: Store Residuals and Predictions
-

**We store the base prediction and residuals in new columns so we can update them iteratively during boosting.**

In [ ]:
train_data['Residuals'] = residuals
train_data['prediction'] = base
train_data.head(10)

Step 11: Define a Custom Tree-Splitting Rule
-
**Instead of using Scikit-learn’s decision trees, we define a manual tree that splits the data based on rules like BMI, smoking status, children and age. This forms the foundation of our model’s learning logic.**

In [ ]:
def tree(person):
    if person['smoker_yes']:
        if person['bmi'] >= 30:
            return 'leaf1'
        else:
            return 'leaf2'
    else:
        if person['sex_male']:
            if person['age'] >= 40:
                return 'leaf3'
            else:
                return 'leaf4'
        else:
            if person['children'] >= 3:
                return 'leaf5'
            else:
                return 'leaf6'


Step 12: Set Boosting Parameters
-
**We define the learning rate (how much we correct each round), number of boosting rounds, and a place to store the corrections for each leaf group.**

In [ ]:
learning_rate = 0.1
num_rounds = 100
leaf_corrections = {'leaf1': [], 'leaf2': [], 'leaf3': [], 'leaf4': [],'leaf5':[], 'leaf6':[]}

train_data = train_data.copy()
train_data['prediction'] = base
train_data['Residuals'] = train_data['charges'] - train_data['prediction']

Step 13: Boosting Loop (Training)
-
**Now we enter the heart of the model — a loop that:**
- Groups data by leaf
- Calculates the mean residual per leaf
- Applies a scaled correction to each group
- Updates predictions and residuals
  
**We also store the training MAE at each round to track learning progress.**

In [ ]:
mae_per_round = []

for _ in range(num_rounds):
    step_corrections = {}

    for leaf in leaf_corrections:
        group = train_data[train_data.apply(tree, axis=1) == leaf]
        if not group.empty:
            mean_residual = group['Residuals'].mean()
            correction = learning_rate * mean_residual
        else:
            correction = 0
        step_corrections[leaf] = correction
        leaf_corrections[leaf].append(correction)


    for idx, row in train_data.iterrows():
        leaf = tree(row)
        train_data.at[idx, 'prediction'] += step_corrections[leaf]

    train_data['Residuals'] = train_data['charges'] - train_data['prediction']
    mae = mean_absolute_error(train_data['charges'], train_data['prediction'])
    mae_per_round.append(mae)
    print(f"Round {_ + 1} MAE: {mae:.2f}")




Step 14: Apply Corrections to Test Data
-
**We apply the same set of corrections from training to the test data, round-by-round, without ever looking at the test targets. This simulates how we’d use the model on new, unseen data.**

In [ ]:
test_data = test_data.copy()
test_data['leaf'] = test_data.apply(tree, axis=1)
test_data['prediction'] = base

test_mae_per_round = []

for i in range(num_rounds):
    for idx, row in test_data.iterrows():
        leaf = row['leaf']
        test_data.at[idx, 'prediction'] += leaf_corrections[leaf][i]

    mae = mean_absolute_error(test_data['charges'], test_data['prediction'])
    test_mae_per_round.append(mae)

    print(f"Round {i+1} Test MAE: {mae:.2f}")


Step 15: Track Best Test Round
-
**We find out at which round our test error was lowest — this helps us detect overfitting and choose the best number of boosting rounds.**


In [ ]:
best_train_round = np.argmin(mae_per_round) + 1
best_train_mae = mae_per_round[best_train_round - 1]
best_test_round = np.argmin(test_mae_per_round) + 1
best_test_mae = test_mae_per_round[best_test_round - 1]

print(f"\n Best Training MAE: {best_train_mae:.2f} at round {best_train_round}")
print(f"\n Best Test MAE: {best_test_mae:.2f} at round {best_test_round}")


Step 16: Plot MAE Over Boosting Rounds
-
**Here we visualize how error changes over time. A good model will show training and test error decreasing together, then test error stabilizing or rising (indicating overfitting).**


In [ ]:
plt.plot(range(1, len(mae_per_round)+1), mae_per_round, label='Train MAE')
plt.plot(range(1, len(test_mae_per_round)+1), test_mae_per_round, label='Test MAE')
plt.axvline(np.argmin(test_mae_per_round)+1, color='gray', linestyle='--', label='Best Test Round')
plt.xlabel("Boosting Round")
plt.ylabel("MAE")
plt.title("Train vs Test MAE per Round")
plt.legend()
plt.grid(True)
plt.show()

Step 17: R² Score
-

**According to the R² Score, the model performs with 80%, which is adequete for a simple hand-build algorithm. This gives us a more intuitive understanding of model performance.**

In [ ]:

r2 = r2_score(test_data['charges'], test_data['prediction'])
print(f"R² Score: {np.round(r2*100):.4f}")

Step 18: Train Sklearn Gradient Boosting Regressor
-
**Now we switch to Scikit-learn’s `GradientBoostingRegressor` to compare performance. This model uses actual decision trees and automated optimization under the hood.**

In [ ]:
features = [col for col in df_encoded.columns if col != 'charges']
X_train = train_data[features]
y_train = train_data['charges']
X_test = test_data[features]
y_test = test_data['charges']

gbr = GradientBoostingRegressor(n_estimators=100, learning_rate=0.05, max_depth=3, random_state=42)
gbr.fit(X_train, y_train)
gbr_preds = gbr.predict(X_test)

mae = mean_absolute_error(y_test, gbr_preds)
r2 = r2_score(y_test, gbr_preds)
print(f"Gradient Boosting Regressor MAE: {mae:.2f}")
print(f"Gradient Boosting Regressor R² Score: {r2*100:.4f}")


Step 19: Evaluate and Visualize Sklearn Model
-

**We calculate the MAE and R² of the Scikit-learn model and plot its predictions against actual values — just like we did for our custom model.**

In [ ]:
plt.figure(figsize=(10, 6))

# Custom model predictions (already stored in test_data['prediction'])
plt.scatter(y_test, test_data['prediction'], alpha=0.6, label='Custom Model', color='blue')

# Scikit-learn predictions
plt.scatter(y_test, gbr_preds, alpha=0.6, label='Sklearn GBR', color='orange')

plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()],
         color='red', linestyle='--', linewidth=2, label='Perfect Prediction Line')

plt.xlabel('Actual Charges')
plt.ylabel('Predicted Charges')
plt.title('Custom Boosting vs Sklearn GBR: Predicted vs Actual')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()

Final Thoughts
-
**We built a gradient boosting model from scratch and compared it with Scikit-learn’s implementation. While simple, our custom model performed surprisingly well and gave us valuable insight into how boosting improves predictions round by round.**

**Great for anyone looking to learn boosting from the ground up!**
